# SciTeX Linter — Notebook Usage

This notebook demonstrates using the scitex-linter Python API programmatically.

## 1. Lint Source Code Directly

In [ ]:
from scitex_linter.checker import lint_source
from scitex_linter.formatter import format_issue

# A script with several SciTeX anti-patterns
bad_code = """\
import matplotlib.pyplot as plt
import numpy as np

def main():
    data = np.load("/tmp/data.npy")
    fig, ax = plt.subplots()
    ax.plot(data)
    plt.savefig("plot.png")
    plt.show()

main()
"""

issues = lint_source(bad_code, "example.py")
for issue in issues:
    print(format_issue(issue, "example.py", color=False))
    print()

## 2. Lint a File on Disk

In [ ]:
from scitex_linter.checker import lint_file
from scitex_linter.formatter import format_issue, format_summary

filepath = "sample_bad.py"
issues = lint_file(filepath)

for issue in issues:
    print(format_issue(issue, filepath, color=False))
    print()

print(format_summary(issues, filepath, color=False))

## 3. JSON Output

In [ ]:
import json
from scitex_linter.formatter import to_json

result = to_json(issues, filepath)
print(json.dumps(result, indent=2))

## 4. Auto-Fix with the Fixer

In [ ]:
from scitex_linter.fixer import fix_source

code_with_savefig = """\
import scitex as stx
import numpy as np

@stx.session
def main():
    data = np.load("./data.npy")
    fig, ax = stx.plt.subplots()
    ax.plot(data)
    fig.savefig("./plot.png", dpi=300)
    return 0

if __name__ == "__main__":
    main()
"""

fixed = fix_source(code_with_savefig, "example.py")
print(fixed)

## 5. Browse All Rules

In [ ]:
from scitex_linter.rules import ALL_RULES

# Group by category
by_cat = {}
for rule in ALL_RULES.values():
    by_cat.setdefault(rule.category, []).append(rule)

for cat, rules in sorted(by_cat.items()):
    print(f"\n=== {cat.upper()} ({len(rules)} rules) ===")
    for r in sorted(rules, key=lambda x: x.id):
        print(f"  {r.id}  [{r.severity}]  {r.message}")

## 6. A Clean Script (Zero Issues)

In [ ]:
clean_code = """\
import scitex as stx

@stx.session
def main(
    data_path="./data.csv",
    threshold=0.5,
    CONFIG=stx.session.INJECTED,
    plt=stx.session.INJECTED,
    COLORS=stx.session.INJECTED,
    rngg=stx.session.INJECTED,
    logger=stx.session.INJECTED,
):
    df = stx.io.load(data_path)
    stx.io.save(df, "./output.csv")
    return 0

if __name__ == "__main__":
    main()
"""

issues = lint_source(clean_code, "clean.py")
print(f"Issues found: {len(issues)}")
if not issues:
    print("Zero lint issues. Fully reproducible.")